# WiSig — Module 2 (Final streamlined, 4-protocol sweep)

This notebook keeps the **final streamlined gate** unchanged and upgrades only the
**evaluation harness**:

- sweep **4 representative protocol scenarios**
- keep the original streamlined `S_id + S_dom` mechanism
- add **boundary-error decomposition**
- export **per-protocol summaries** for stability analysis

Streamlined gate:
- **S_id**: `Fusion_lam0.5 = z(EmbMaha) + 0.5 * z(Energy)`
- **S_dom**: `V1_mixNLL`
- **Gate**:
  - if `S_id < τ_id` ⇒ **Unknown**
  - else if `S_dom > τ_dom` ⇒ **Known-Drift**
  - else ⇒ **Known-Stable**

Representative protocol combinations:
- `RX9-3_TX4-2`
- `RX9-3_TX2-4`
- `RX6-6_TX3-3`
- `RX3-9_TX2-4`

Added boundary errors:
- `false_drift_stable`   : Stable → Drift
- `false_unknown_stable` : Stable → Unknown
- `false_drift_unknown`  : Unknown → Drift
- `false_stable_unknown` : Unknown → Stable
- `false_unknown_drift`  : Drift → Unknown

This notebook is intended to answer:
1. Is the streamlined gate stable across representative protocol regimes?
2. Does it fail mainly by over-rejecting unknown, or by missing known-drift?
3. Is it a better middle-ground than v3 / SR1 for downstream Module 3 coupling?


In [1]:
import os, json, time, itertools
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4
TX_SPLIT_REPEATS = 5

# ---- protocol libraries ----
RX_PROTOCOL_LIBRARY = {
    "3-9": dict(source_rx_num=3, drift_rx_num=9),
    "6-6": dict(source_rx_num=6, drift_rx_num=6),
    "9-3": dict(source_rx_num=9, drift_rx_num=3),
}
TX_PROTOCOL_LIBRARY = {
    "2-4": dict(known_tx_num=2, unknown_tx_num=4),
    "3-3": dict(known_tx_num=3, unknown_tx_num=3),
    "4-2": dict(known_tx_num=4, unknown_tx_num=2),
}

# 4 representative scenarios used in the recent analysis thread.
SELECTED_PROTOCOL_COMBOS = [
    ("9-3", "4-2"),
    ("9-3", "2-4"),
    ("6-6", "3-3"),
    ("3-9", "2-4"),
]

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signals ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- dims ----
D_DIM = 32

# ---- Sid fusion ----
FUSION_LAM = 0.5

# ---- calibration targets ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05

# ---- joint router diagnostic ----
ROUTER_CAL_FRAC = 0.20
ROUTER_MIN_CAL_PER_POOL = 256
ROUTER_MAX_ITER = 2000
ROUTER_TOP2_TEMP = 0.20
JOINT_ROUTER_MODE = "top2soft"   # {"hardk", "top2soft"}

# NOTE:
# The joint router below is calibrated on labeled state-holdout subsets
# sampled from Stable / Drift / Unknown pools inside each fold.
# This is a decision-layer diagnostic, not the final paper protocol.

# ---- outputs ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"../owdc_results/results_wisig_module2_joint_router_protocols/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED,
    TX_TOTAL_USE=TX_TOTAL_USE,
    RX_TOTAL_USE=RX_TOTAL_USE,
    DAY_TOTAL_USE=DAY_TOTAL_USE,
    TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    RX_PROTOCOL_LIBRARY=RX_PROTOCOL_LIBRARY,
    TX_PROTOCOL_LIBRARY=TX_PROTOCOL_LIBRARY,
    SELECTED_PROTOCOL_COMBOS=SELECTED_PROTOCOL_COMBOS,
    TRAIN_DATES=TRAIN_DATES,
    TEST_DATES_RX=TEST_DATES_RX,
    TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ,
    D_FROM=D_FROM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES,
    DROPOUT=DROPOUT,
    D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM,
    FRR_TARGET=FRR_TARGET,
    FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    ROUTER_CAL_FRAC=ROUTER_CAL_FRAC,
    ROUTER_MIN_CAL_PER_POOL=ROUTER_MIN_CAL_PER_POOL,
    ROUTER_MAX_ITER=ROUTER_MAX_ITER,
    ROUTER_TOP2_TEMP=ROUTER_TOP2_TEMP,
    JOINT_ROUTER_MODE=JOINT_ROUTER_MODE,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)


Device: cuda
RUN_DIR: ../owdc_results/results_wisig_module2_joint_router_protocols/run_20260324_142641


In [2]:
# =========================
# Data utilities
# =========================
def get_idx(lst, val): 
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [3]:
# =========================
# Protocol suite construction
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num})."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for local_id, idx in enumerate(order[:n_splits], start=1):
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append({
            "split_id": local_id,
            "known": known,
            "unknown": unknown,
        })
    return tx_splits

def build_protocol_specs(
    tx_use, rx_use,
    protocol_combos,
    rx_protocol_library, tx_protocol_library,
    tx_split_repeats, seed=42
):
    protocol_specs = []
    for combo_idx, (rx_tag, tx_tag) in enumerate(protocol_combos, start=1):
        if rx_tag not in rx_protocol_library:
            raise KeyError(f"Unknown RX protocol tag: {rx_tag}")
        if tx_tag not in tx_protocol_library:
            raise KeyError(f"Unknown TX protocol tag: {tx_tag}")

        rx_cfg = dict(rx_protocol_library[rx_tag])
        tx_cfg = dict(tx_protocol_library[tx_tag])

        src_n = int(rx_cfg["source_rx_num"])
        drift_n = int(rx_cfg["drift_rx_num"])
        known_n = int(tx_cfg["known_tx_num"])
        unknown_n = int(tx_cfg["unknown_tx_num"])

        if src_n + drift_n != len(rx_use):
            raise ValueError(
                f"RX protocol {rx_tag} expects source+drift={src_n + drift_n}, "
                f"but len(RX_USE)={len(rx_use)}."
            )
        if known_n + unknown_n != len(tx_use):
            raise ValueError(
                f"TX protocol {tx_tag} expects known+unknown={known_n + unknown_n}, "
                f"but len(TX_USE)={len(tx_use)}."
            )

        rng_rx = np.random.RandomState(seed + 1000 * combo_idx + 17 * src_n + drift_n)
        source_rxs = list(rng_rx.choice(list(rx_use), size=src_n, replace=False))
        source_rxs.sort()
        drift_rx = [r for r in rx_use if r not in source_rxs]

        tx_splits = build_unique_tx_splits(
            tx_use=tx_use,
            known_tx_num=known_n,
            n_splits=tx_split_repeats,
            seed=seed + 5000 + 97 * combo_idx,
        )

        protocol_specs.append(dict(
            protocol_tag=f"RX{rx_tag}_TX{tx_tag}",
            rx_protocol=rx_tag,
            tx_protocol=tx_tag,
            source_rx_num=src_n,
            drift_rx_num=drift_n,
            known_tx_num=known_n,
            unknown_tx_num=unknown_n,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            tx_splits=tx_splits,
        ))
    return protocol_specs

PROTOCOL_SPECS = build_protocol_specs(
    TX_USE, RX_USE,
    SELECTED_PROTOCOL_COMBOS,
    RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
    tx_split_repeats=TX_SPLIT_REPEATS,
    seed=SEED + 777,
)

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_SPECS, f, indent=2)

print("Selected protocol combinations:", len(PROTOCOL_SPECS))
for spec in PROTOCOL_SPECS:
    print(
        f"[{spec['protocol_tag']}] "
        f"SOURCE_RXS={spec['source_rxs']} | DRIFT_RX={spec['drift_rx']} | "
        f"TXsplits={len(spec['tx_splits'])}"
    )


TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
Selected protocol combinations: 4
[RX9-3_TX4-2] SOURCE_RXS=[np.str_('14-7'), np.str_('18-2'), np.str_('2-1'), np.str_('2-19'), np.str_('20-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7'), np.str_('8-8')] | DRIFT_RX=['1-1', '1-19', '19-2'] | TXsplits=5
[RX9-3_TX2-4] SOURCE_RXS=[np.str_('1-1'), np.str_('1-19'), np.str_('14-7'), np.str_('18-2'), np.str_('2-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7'), np.str_('8-8')] | DRIFT_RX=['19-2', '2-19', '20-1'] | TXsplits=5
[RX6-6_TX3-3] SOURCE_RXS=[np.str_('18-2'), np.str_('2-19'), np.str_('20-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7')] | DRIFT_RX=['1-1', '1-19', '14-7', '19-2', '2-1', '8-8'] | TXsplits=5
[RX3-9_TX2-4] SOURCE_RXS=[np.str_('1-19'), np.str_('14-7'), np.str_('7-14')] | DRIFT_RX=['1-1

In [4]:
# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [5]:
# =========================
# Train / inference helpers
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): 
        return self.X.shape[0]
    def __getitem__(self, i): 
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x/(np.linalg.norm(x,axis=axis,keepdims=True)+eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y==k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z,axis=1) @ P.T

def sid_EmbMaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

In [6]:
# =========================
# Domain (Sdom): V1_mixNLL
# =========================
def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps=[]
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum() + 1e-12) - np.log(R)
        out[i] = float(-loglik)  # higher => more drift
    return out

In [7]:
# =========================
# Build datasets
# =========================
def build_source_train(compact_dataset, KNOWN_TX, source_rxs):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in source_rxs:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    X = np.concatenate(X_list,0).astype(np.float32)
    y = np.concatenate(y_list,0).astype(np.int64)
    D = np.concatenate(D_list,0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list,0).astype(np.int64)
    return X,y,D,rx_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, D_list = [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); D_list.append(Df)
    X = np.concatenate(X_list,0).astype(np.float32)
    D = np.concatenate(D_list,0).astype(np.float32)
    return X, D

In [8]:
# =========================
# Joint router + plots
# =========================
def fit_classwise_dom_stats(Sdom_train, y_train, K, min_std=1e-3):
    Sdom_train = np.asarray(Sdom_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64)

    global_mu = float(np.mean(Sdom_train))
    global_std = float(max(np.std(Sdom_train), min_std))

    dom_mu = np.full((K,), global_mu, dtype=np.float32)
    dom_std = np.full((K,), global_std, dtype=np.float32)

    for k in range(K):
        vals = Sdom_train[y_train == k]
        if vals.size >= 8:
            dom_mu[k] = float(np.mean(vals))
            dom_std[k] = float(max(np.std(vals), min_std))
    return dom_mu, dom_std, global_mu, global_std

def normalize_dom_matrix_by_class(Sdom_allk, dom_mu, dom_std, global_mu, global_std):
    Sdom_allk = np.asarray(Sdom_allk, dtype=np.float32)
    K = Sdom_allk.shape[1]
    out = np.zeros_like(Sdom_allk, dtype=np.float32)
    for k in range(K):
        mu = float(dom_mu[k]) if k < len(dom_mu) else float(global_mu)
        sd = float(dom_std[k]) if k < len(dom_std) else float(global_std)
        out[:, k] = (Sdom_allk[:, k] - mu) / max(sd, 1e-6)
    return out.astype(np.float32)

def sdom_V1_mixNLL_allk(D_eval, K, models_kr, fallback_k, source_rx_ids):
    D_eval = np.asarray(D_eval, dtype=np.float32)
    N = D_eval.shape[0]
    R = len(source_rx_ids)
    out = np.zeros((N, K), dtype=np.float32)
    for k in range(K):
        logps = []
        for rxid in source_rx_ids:
            mu, Sinv, logdet = models_kr.get((k, rxid), fallback_k[k])
            logps.append(logpdf_gaussian(D_eval, mu, Sinv, logdet).astype(np.float32))
        logps = np.stack(logps, axis=1)  # [N, R]
        m = np.max(logps, axis=1, keepdims=True)
        loglik = m[:, 0] + np.log(np.exp(logps - m).sum(axis=1) + 1e-12) - np.log(R)
        out[:, k] = (-loglik).astype(np.float32)
    return out

def gather_hard_class_scores(score_allk, k_hat):
    score_allk = np.asarray(score_allk, dtype=np.float32)
    k_hat = np.asarray(k_hat, dtype=np.int64)
    return score_allk[np.arange(len(k_hat)), k_hat].astype(np.float32)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float32)
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.clip(np.sum(e, axis=axis, keepdims=True), 1e-12, None)

def aggregate_top2_scores(score_allk, id_score_allk, temp=0.20):
    score_allk = np.asarray(score_allk, dtype=np.float32)
    id_score_allk = np.asarray(id_score_allk, dtype=np.float32)

    top2_idx = np.argsort(id_score_allk, axis=1)[:, -2:][:, ::-1]
    top2_id_scores = np.take_along_axis(id_score_allk, top2_idx, axis=1)
    weights = softmax_np(top2_id_scores / max(float(temp), 1e-6), axis=1)
    top2_scores = np.take_along_axis(score_allk, top2_idx, axis=1)
    agg = np.sum(weights * top2_scores, axis=1)
    return agg.astype(np.float32), top2_idx.astype(np.int64), weights.astype(np.float32)

def build_router_features(Sid, Sdom_raw, Sdrift_norm):
    return np.stack(
        [
            np.asarray(Sid, dtype=np.float32),
            np.asarray(Sdom_raw, dtype=np.float32),
            np.asarray(Sdrift_norm, dtype=np.float32),
        ],
        axis=1,
    ).astype(np.float32)

def split_cal_eval_indices(n, frac, rng, min_cal=256):
    if n < 4:
        idx = np.arange(n)
        cut = max(1, n // 2)
        return idx[:cut], idx[cut:]
    n_cal = max(1, int(round(frac * n)))
    if n >= 2 * min_cal:
        n_cal = max(n_cal, min_cal)
    n_cal = min(n_cal, n - 1)
    perm = rng.permutation(n)
    return perm[:n_cal], perm[n_cal:]

def fit_joint_router_multinomial(X_cal, y_cal, max_iter=2000):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(np.asarray(X_cal, dtype=np.float32))
    clf = LogisticRegression(
        multi_class="multinomial",
        class_weight="balanced",
        max_iter=max_iter,
        random_state=SEED,
    )
    clf.fit(Xs, np.asarray(y_cal, dtype=np.int64))
    return scaler, clf

def predict_joint_router(X, scaler, clf):
    Xs = scaler.transform(np.asarray(X, dtype=np.float32))
    probs = clf.predict_proba(Xs).astype(np.float32)
    pred = np.argmax(probs, axis=1).astype(np.int64)
    return pred, probs

def calibrate_thresholds(Sid_stable, Sdom_stable, frr_target=0.05, fdr_target=0.05):
    tau_id = float(np.quantile(Sid_stable, frr_target))
    tau_dom = float(np.quantile(Sdom_stable, 1.0 - fdr_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def plot_gate_scatter(Sid, Sdom, gt, tau_id, tau_dom, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdom[idx], s=3, c=gt[idx], alpha=0.5)
    plt.axvline(tau_id, linestyle="--")
    plt.axhline(tau_dom, linestyle="--")
    plt.xlabel("S_id")
    plt.ylabel("S_dom_raw")
    plt.title("Baseline hard gate (held-out eval)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_joint_scatter(Sid, Sdrift, gt, out_png, max_points=20000):
    n = len(Sid)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6,5))
    plt.scatter(Sid[idx], Sdrift[idx], s=3, c=gt[idx], alpha=0.5)
    plt.xlabel("S_id")
    plt.ylabel("S_drift_norm")
    plt.title("Joint router features (held-out eval)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5,4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.yticks([0,1,2], ["Stable","Drift","Unknown"])
    plt.xlabel("Pred")
    plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

def compute_state_metrics(pred_A, pred_B, pred_C, pred_D, pred_E, pred_F):
    gt = np.concatenate([
        np.zeros_like(pred_A, dtype=np.int64),
        np.ones_like(pred_B, dtype=np.int64),
        np.ones_like(pred_C, dtype=np.int64),
        np.full_like(pred_D, 2, dtype=np.int64),
        np.full_like(pred_E, 2, dtype=np.int64),
        np.full_like(pred_F, 2, dtype=np.int64),
    ])
    pred = np.concatenate([pred_A, pred_B, pred_C, pred_D, pred_E, pred_F])
    cm = confusion_matrix(gt, pred, labels=[0,1,2])

    stable_acc_gate = float((pred_A == 0).mean())
    drift_acc_rx = float((pred_B == 1).mean())
    drift_acc_day = float((pred_C == 1).mean())
    drift_preds = np.concatenate([pred_B, pred_C])
    unknown_preds = np.concatenate([pred_D, pred_E, pred_F])

    drift_acc_all = float((drift_preds == 1).mean())
    unknown_reject_in = float((pred_D == 2).mean())
    unknown_reject_rx = float((pred_E == 2).mean())
    unknown_reject_day = float((pred_F == 2).mean())
    unknown_reject_all = float((unknown_preds == 2).mean())

    FRR_stable = float((pred_A != 0).mean())
    false_drift_stable = float((pred_A == 1).mean())
    false_unknown_stable = float((pred_A == 2).mean())

    FAR_unknown_accept = float((unknown_preds != 2).mean())
    FAR_unknown_accept_in = float((pred_D != 2).mean())
    FAR_unknown_accept_rx = float((pred_E != 2).mean())
    FAR_unknown_accept_day = float((pred_F != 2).mean())

    false_drift_unknown = float((unknown_preds == 1).mean())
    false_drift_unknown_in = float((pred_D == 1).mean())
    false_drift_unknown_rx = float((pred_E == 1).mean())
    false_drift_unknown_day = float((pred_F == 1).mean())

    false_stable_unknown = float((unknown_preds == 0).mean())
    false_stable_unknown_in = float((pred_D == 0).mean())
    false_stable_unknown_rx = float((pred_E == 0).mean())
    false_stable_unknown_day = float((pred_F == 0).mean())

    miss_drift_rx = float((pred_B == 0).mean())
    miss_drift_day = float((pred_C == 0).mean())
    miss_drift_all = float((np.sum(pred_B == 0) + np.sum(pred_C == 0)) / (len(pred_B) + len(pred_C)))

    false_unknown_drift_rx = float((pred_B == 2).mean())
    false_unknown_drift_day = float((pred_C == 2).mean())
    false_unknown_drift_all = float((np.sum(pred_B == 2) + np.sum(pred_C == 2)) / (len(pred_B) + len(pred_C)))

    metrics = dict(
        stable_acc_gate=stable_acc_gate,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        unknown_reject_in=unknown_reject_in,
        unknown_reject_rx=unknown_reject_rx,
        unknown_reject_day=unknown_reject_day,
        unknown_reject_all=unknown_reject_all,
        FRR_stable=FRR_stable,
        false_drift_stable=false_drift_stable,
        false_unknown_stable=false_unknown_stable,
        FAR_unknown_accept=FAR_unknown_accept,
        FAR_unknown_accept_in=FAR_unknown_accept_in,
        FAR_unknown_accept_rx=FAR_unknown_accept_rx,
        FAR_unknown_accept_day=FAR_unknown_accept_day,
        false_drift_unknown=false_drift_unknown,
        false_drift_unknown_in=false_drift_unknown_in,
        false_drift_unknown_rx=false_drift_unknown_rx,
        false_drift_unknown_day=false_drift_unknown_day,
        false_stable_unknown=false_stable_unknown,
        false_stable_unknown_in=false_stable_unknown_in,
        false_stable_unknown_rx=false_stable_unknown_rx,
        false_stable_unknown_day=false_stable_unknown_day,
        miss_drift_rx=miss_drift_rx,
        miss_drift_day=miss_drift_day,
        miss_drift_all=miss_drift_all,
        false_unknown_drift_rx=false_unknown_drift_rx,
        false_unknown_drift_day=false_unknown_drift_day,
        false_unknown_drift_all=false_unknown_drift_all,
    )
    return metrics, cm, gt, pred

def prefix_keys(d, prefix):
    return {f"{prefix}{k}": v for k, v in d.items()}

In [9]:
# =========================
# Main loop
# =========================
def run_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx):
    protocol_dir = os.path.join(RUN_DIR, protocol_tag)
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "protocol_tag": protocol_tag,
                "rx_protocol": rx_protocol_tag,
                "tx_protocol": tx_protocol_tag,
                "KNOWN_TX": KNOWN_TX,
                "UNKNOWN_TX": UNKNOWN_TX,
                "SOURCE_RXS": source_rxs,
                "DRIFT_RX": drift_rx,
            },
            f,
            indent=2,
        )

    # source (known TX, source RX, source day)
    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs)
    K = len(KNOWN_TX)

    # drift sets (known TX)
    X_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, source_rxs, TEST_DATES_DAY)

    # unknown sets (unknown TX)
    X_u_in, D_u_in = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_u_drRX, D_u_drRX = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_u_drDAY, D_u_drDAY = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id + len(source_rxs))
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        # val split
        rng = np.random.RandomState(SEED + 1000*split_id + 100*len(source_rxs) + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1*len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ArrayDataset(X_src[idx_val], y_src[idx_val]), batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        for ep in range(1, EPOCHS + 1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss)
            hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss)
            hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        # stable A
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        acc = float((np.argmax(logits_A, 1) == y_src[idx_te]).mean())
        D_A = D_src[idx_te]

        # fit train-time stats
        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        mu_z, var_z = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        P = prototypes(Z_tr, y_src[idx_train], K)

        # eval pools
        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        # Sid (keep streamlined score construction unchanged)
        SidA_emb = sid_EmbMaha(Z_A, mu_z, var_z)
        SidB_emb = sid_EmbMaha(Z_B, mu_z, var_z)
        SidC_emb = sid_EmbMaha(Z_C, mu_z, var_z)
        SidD_emb = sid_EmbMaha(Z_D, mu_z, var_z)
        SidE_emb = sid_EmbMaha(Z_E, mu_z, var_z)
        SidF_emb = sid_EmbMaha(Z_F, mu_z, var_z)

        SidA_en = sid_Energy(logits_A)
        SidB_en = sid_Energy(logits_B)
        SidC_en = sid_Energy(logits_C)
        SidD_en = sid_Energy(logits_D)
        SidE_en = sid_Energy(logits_E)
        SidF_en = sid_Energy(logits_F)

        zA_emb = zscore(SidA_emb, SidA_emb)
        zB_emb = zscore(SidB_emb, SidA_emb)
        zC_emb = zscore(SidC_emb, SidA_emb)
        zD_emb = zscore(SidD_emb, SidA_emb)
        zE_emb = zscore(SidE_emb, SidA_emb)
        zF_emb = zscore(SidF_emb, SidA_emb)

        zA_en = zscore(SidA_en, SidA_en)
        zB_en = zscore(SidB_en, SidA_en)
        zC_en = zscore(SidC_en, SidA_en)
        zD_en = zscore(SidD_en, SidA_en)
        zE_en = zscore(SidE_en, SidA_en)
        zF_en = zscore(SidF_en, SidA_en)

        Sid_A = (zA_emb + FUSION_LAM * zA_en).astype(np.float32)
        Sid_B = (zB_emb + FUSION_LAM * zB_en).astype(np.float32)
        Sid_C = (zC_emb + FUSION_LAM * zC_en).astype(np.float32)
        Sid_D = (zD_emb + FUSION_LAM * zD_en).astype(np.float32)
        Sid_E = (zE_emb + FUSION_LAM * zE_en).astype(np.float32)
        Sid_F = (zF_emb + FUSION_LAM * zF_en).astype(np.float32)

        # identity candidates for raw / soft domain scoring
        CosA = cosine_to_proto(Z_A, P)
        CosB = cosine_to_proto(Z_B, P)
        CosC = cosine_to_proto(Z_C, P)
        CosD = cosine_to_proto(Z_D, P)
        CosE = cosine_to_proto(Z_E, P)
        CosF = cosine_to_proto(Z_F, P)

        kA = np.argmax(CosA, 1).astype(np.int64)
        kB = np.argmax(CosB, 1).astype(np.int64)
        kC = np.argmax(CosC, 1).astype(np.int64)
        kD = np.argmax(CosD, 1).astype(np.int64)
        kE = np.argmax(CosE, 1).astype(np.int64)
        kF = np.argmax(CosF, 1).astype(np.int64)

        source_rx_ids = sorted({RX_USE.index(r) for r in source_rxs})
        models_kr, fb = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)

        # train-time classwise domain normalization
        Sdom_tr_true = sdom_V1_mixNLL(D_src[idx_train], y_src[idx_train], models_kr, fb, source_rx_ids)
        dom_mu, dom_std, dom_global_mu, dom_global_std = fit_classwise_dom_stats(Sdom_tr_true, y_src[idx_train], K, min_std=1e-3)

        # classwise domain scores for each eval pool
        Sdom_allk_A = sdom_V1_mixNLL_allk(D_A, K, models_kr, fb, source_rx_ids)
        Sdom_allk_B = sdom_V1_mixNLL_allk(D_drRX, K, models_kr, fb, source_rx_ids)
        Sdom_allk_C = sdom_V1_mixNLL_allk(D_drDAY, K, models_kr, fb, source_rx_ids)
        Sdom_allk_D = sdom_V1_mixNLL_allk(D_u_in, K, models_kr, fb, source_rx_ids)
        Sdom_allk_E = sdom_V1_mixNLL_allk(D_u_drRX, K, models_kr, fb, source_rx_ids)
        Sdom_allk_F = sdom_V1_mixNLL_allk(D_u_drDAY, K, models_kr, fb, source_rx_ids)

        Sdrift_allk_A = normalize_dom_matrix_by_class(Sdom_allk_A, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_B = normalize_dom_matrix_by_class(Sdom_allk_B, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_C = normalize_dom_matrix_by_class(Sdom_allk_C, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_D = normalize_dom_matrix_by_class(Sdom_allk_D, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_E = normalize_dom_matrix_by_class(Sdom_allk_E, dom_mu, dom_std, dom_global_mu, dom_global_std)
        Sdrift_allk_F = normalize_dom_matrix_by_class(Sdom_allk_F, dom_mu, dom_std, dom_global_mu, dom_global_std)

        # baseline hard-k domain scores
        Sdom_hard_A = gather_hard_class_scores(Sdom_allk_A, kA)
        Sdom_hard_B = gather_hard_class_scores(Sdom_allk_B, kB)
        Sdom_hard_C = gather_hard_class_scores(Sdom_allk_C, kC)
        Sdom_hard_D = gather_hard_class_scores(Sdom_allk_D, kD)
        Sdom_hard_E = gather_hard_class_scores(Sdom_allk_E, kE)
        Sdom_hard_F = gather_hard_class_scores(Sdom_allk_F, kF)

        Sdrift_hard_A = gather_hard_class_scores(Sdrift_allk_A, kA)
        Sdrift_hard_B = gather_hard_class_scores(Sdrift_allk_B, kB)
        Sdrift_hard_C = gather_hard_class_scores(Sdrift_allk_C, kC)
        Sdrift_hard_D = gather_hard_class_scores(Sdrift_allk_D, kD)
        Sdrift_hard_E = gather_hard_class_scores(Sdrift_allk_E, kE)
        Sdrift_hard_F = gather_hard_class_scores(Sdrift_allk_F, kF)

        # step-2 candidate: top-2 soft aggregation
        Sdom_soft_A, _, _ = aggregate_top2_scores(Sdom_allk_A, CosA, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_B, _, _ = aggregate_top2_scores(Sdom_allk_B, CosB, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_C, _, _ = aggregate_top2_scores(Sdom_allk_C, CosC, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_D, _, _ = aggregate_top2_scores(Sdom_allk_D, CosD, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_E, _, _ = aggregate_top2_scores(Sdom_allk_E, CosE, temp=ROUTER_TOP2_TEMP)
        Sdom_soft_F, _, _ = aggregate_top2_scores(Sdom_allk_F, CosF, temp=ROUTER_TOP2_TEMP)

        Sdrift_soft_A, _, _ = aggregate_top2_scores(Sdrift_allk_A, CosA, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_B, _, _ = aggregate_top2_scores(Sdrift_allk_B, CosB, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_C, _, _ = aggregate_top2_scores(Sdrift_allk_C, CosC, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_D, _, _ = aggregate_top2_scores(Sdrift_allk_D, CosD, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_E, _, _ = aggregate_top2_scores(Sdrift_allk_E, CosE, temp=ROUTER_TOP2_TEMP)
        Sdrift_soft_F, _, _ = aggregate_top2_scores(Sdrift_allk_F, CosF, temp=ROUTER_TOP2_TEMP)

        if JOINT_ROUTER_MODE == "hardk":
            JR_Sdom_A, JR_Sdom_B, JR_Sdom_C = Sdom_hard_A, Sdom_hard_B, Sdom_hard_C
            JR_Sdom_D, JR_Sdom_E, JR_Sdom_F = Sdom_hard_D, Sdom_hard_E, Sdom_hard_F
            JR_Sdr_A, JR_Sdr_B, JR_Sdr_C = Sdrift_hard_A, Sdrift_hard_B, Sdrift_hard_C
            JR_Sdr_D, JR_Sdr_E, JR_Sdr_F = Sdrift_hard_D, Sdrift_hard_E, Sdrift_hard_F
        elif JOINT_ROUTER_MODE == "top2soft":
            JR_Sdom_A, JR_Sdom_B, JR_Sdom_C = Sdom_soft_A, Sdom_soft_B, Sdom_soft_C
            JR_Sdom_D, JR_Sdom_E, JR_Sdom_F = Sdom_soft_D, Sdom_soft_E, Sdom_soft_F
            JR_Sdr_A, JR_Sdr_B, JR_Sdr_C = Sdrift_soft_A, Sdrift_soft_B, Sdrift_soft_C
            JR_Sdr_D, JR_Sdr_E, JR_Sdr_F = Sdrift_soft_D, Sdrift_soft_E, Sdrift_soft_F
        else:
            raise ValueError(f"Unknown JOINT_ROUTER_MODE={JOINT_ROUTER_MODE}")

        # state-holdout calibration for fair diagnostic comparison
        rng_router = np.random.RandomState(SEED + 1000*split_id + 100*len(source_rxs) + fold + 9999)
        cal_A, ev_A = split_cal_eval_indices(len(Sid_A), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        cal_B, ev_B = split_cal_eval_indices(len(Sid_B), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        cal_C, ev_C = split_cal_eval_indices(len(Sid_C), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        cal_D, ev_D = split_cal_eval_indices(len(Sid_D), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        cal_E, ev_E = split_cal_eval_indices(len(Sid_E), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)
        cal_F, ev_F = split_cal_eval_indices(len(Sid_F), ROUTER_CAL_FRAC, rng_router, ROUTER_MIN_CAL_PER_POOL)

        # baseline hard gate calibrated only on stable holdout-cal
        tau_id, tau_dom = calibrate_thresholds(Sid_A[cal_A], Sdom_hard_A[cal_A], FRR_TARGET, FALSE_DRIFT_TARGET)

        pred_A_base = gate_predict(Sid_A[ev_A], Sdom_hard_A[ev_A], tau_id, tau_dom)
        pred_B_base = gate_predict(Sid_B[ev_B], Sdom_hard_B[ev_B], tau_id, tau_dom)
        pred_C_base = gate_predict(Sid_C[ev_C], Sdom_hard_C[ev_C], tau_id, tau_dom)
        pred_D_base = gate_predict(Sid_D[ev_D], Sdom_hard_D[ev_D], tau_id, tau_dom)
        pred_E_base = gate_predict(Sid_E[ev_E], Sdom_hard_E[ev_E], tau_id, tau_dom)
        pred_F_base = gate_predict(Sid_F[ev_F], Sdom_hard_F[ev_F], tau_id, tau_dom)

        # joint router calibrated on labeled state-holdout-cal
        X_cal = np.concatenate([
            build_router_features(Sid_A[cal_A], JR_Sdom_A[cal_A], JR_Sdr_A[cal_A]),
            build_router_features(Sid_B[cal_B], JR_Sdom_B[cal_B], JR_Sdr_B[cal_B]),
            build_router_features(Sid_C[cal_C], JR_Sdom_C[cal_C], JR_Sdr_C[cal_C]),
            build_router_features(Sid_D[cal_D], JR_Sdom_D[cal_D], JR_Sdr_D[cal_D]),
            build_router_features(Sid_E[cal_E], JR_Sdom_E[cal_E], JR_Sdr_E[cal_E]),
            build_router_features(Sid_F[cal_F], JR_Sdom_F[cal_F], JR_Sdr_F[cal_F]),
        ], axis=0)
        y_cal = np.concatenate([
            np.zeros((len(cal_A),), dtype=np.int64),
            np.ones((len(cal_B),), dtype=np.int64),
            np.ones((len(cal_C),), dtype=np.int64),
            np.full((len(cal_D),), 2, dtype=np.int64),
            np.full((len(cal_E),), 2, dtype=np.int64),
            np.full((len(cal_F),), 2, dtype=np.int64),
        ], axis=0)

        router_scaler, router_clf = fit_joint_router_multinomial(X_cal, y_cal, max_iter=ROUTER_MAX_ITER)

        XA = build_router_features(Sid_A[ev_A], JR_Sdom_A[ev_A], JR_Sdr_A[ev_A])
        XB = build_router_features(Sid_B[ev_B], JR_Sdom_B[ev_B], JR_Sdr_B[ev_B])
        XC = build_router_features(Sid_C[ev_C], JR_Sdom_C[ev_C], JR_Sdr_C[ev_C])
        XD = build_router_features(Sid_D[ev_D], JR_Sdom_D[ev_D], JR_Sdr_D[ev_D])
        XE = build_router_features(Sid_E[ev_E], JR_Sdom_E[ev_E], JR_Sdr_E[ev_E])
        XF = build_router_features(Sid_F[ev_F], JR_Sdom_F[ev_F], JR_Sdr_F[ev_F])

        pred_A_joint, probs_A_joint = predict_joint_router(XA, router_scaler, router_clf)
        pred_B_joint, probs_B_joint = predict_joint_router(XB, router_scaler, router_clf)
        pred_C_joint, probs_C_joint = predict_joint_router(XC, router_scaler, router_clf)
        pred_D_joint, probs_D_joint = predict_joint_router(XD, router_scaler, router_clf)
        pred_E_joint, probs_E_joint = predict_joint_router(XE, router_scaler, router_clf)
        pred_F_joint, probs_F_joint = predict_joint_router(XF, router_scaler, router_clf)

        # metrics on the SAME held-out eval slices
        base_metrics, cm_base, gt_eval, pred_eval_base = compute_state_metrics(
            pred_A_base, pred_B_base, pred_C_base, pred_D_base, pred_E_base, pred_F_base
        )
        joint_metrics, cm_joint, _, pred_eval_joint = compute_state_metrics(
            pred_A_joint, pred_B_joint, pred_C_joint, pred_D_joint, pred_E_joint, pred_F_joint
        )

        # score-level diagnostics on held-out eval slices
        auc_unknown_sid = roc_auc(
            np.concatenate([
                np.zeros((len(ev_A),), dtype=np.int64),
                np.ones((len(ev_D),), dtype=np.int64),
            ]),
            np.concatenate([
                -Sid_A[ev_A],
                -Sid_D[ev_D],
            ])
        )
        auc_drift_rx_raw_hard = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_B),), dtype=np.int64)]),
            np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_B[ev_B]])
        )
        auc_drift_day_raw_hard = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_C),), dtype=np.int64)]),
            np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_C[ev_C]])
        )
        auc_drift_rx_router = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_B),), dtype=np.int64)]),
            np.concatenate([JR_Sdr_A[ev_A], JR_Sdr_B[ev_B]])
        )
        auc_drift_day_router = roc_auc(
            np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_C),), dtype=np.int64)]),
            np.concatenate([JR_Sdr_A[ev_A], JR_Sdr_C[ev_C]])
        )

        p_unknown_joint = np.concatenate([probs_A_joint[:, 2], probs_D_joint[:, 2]])
        y_unknown_joint = np.concatenate([np.zeros((len(ev_A),), dtype=np.int64), np.ones((len(ev_D),), dtype=np.int64)])
        auc_unknown_jointprob = roc_auc(y_unknown_joint, p_unknown_joint)

        # plots
        Sid_all_eval = np.concatenate([Sid_A[ev_A], Sid_B[ev_B], Sid_C[ev_C], Sid_D[ev_D], Sid_E[ev_E], Sid_F[ev_F]])
        Sdom_all_eval = np.concatenate([Sdom_hard_A[ev_A], Sdom_hard_B[ev_B], Sdom_hard_C[ev_C], Sdom_hard_D[ev_D], Sdom_hard_E[ev_E], Sdom_hard_F[ev_F]])
        Sdrift_all_eval = np.concatenate([JR_Sdr_A[ev_A], JR_Sdr_B[ev_B], JR_Sdr_C[ev_C], JR_Sdr_D[ev_D], JR_Sdr_E[ev_E], JR_Sdr_F[ev_F]])
        gt_all_eval = gt_eval.copy()

        plot_gate_scatter(Sid_all_eval, Sdom_all_eval, gt_all_eval, tau_id, tau_dom, os.path.join(fold_dir, "baseline_gate_scatter_eval.png"))
        plot_joint_scatter(Sid_all_eval, Sdrift_all_eval, gt_all_eval, os.path.join(fold_dir, "joint_router_scatter_eval.png"))
        plot_confmat(cm_base, os.path.join(fold_dir, "baseline_confmat_eval.png"), "Baseline hard gate (held-out eval)")
        plot_confmat(cm_joint, os.path.join(fold_dir, "joint_confmat_eval.png"), f"Joint router [{JOINT_ROUTER_MODE}] (held-out eval)")

        metrics = dict(
            protocol_tag=protocol_tag,
            rx_protocol=rx_protocol_tag,
            tx_protocol=tx_protocol_tag,
            split=split_id,
            fold=fold,
            acc=acc,
            router_mode=JOINT_ROUTER_MODE,
            router_cal_frac=ROUTER_CAL_FRAC,
            router_cal_n_stable=int(len(cal_A)),
            router_cal_n_drift=int(len(cal_B) + len(cal_C)),
            router_cal_n_unknown=int(len(cal_D) + len(cal_E) + len(cal_F)),
            tau_id=float(tau_id),
            tau_dom=float(tau_dom),
            auc_unknown_sid=float(auc_unknown_sid),
            auc_unknown_jointprob=float(auc_unknown_jointprob),
            auc_drift_rx_raw_hard=float(auc_drift_rx_raw_hard),
            auc_drift_day_raw_hard=float(auc_drift_day_raw_hard),
            auc_drift_rx_router=float(auc_drift_rx_router),
            auc_drift_day_router=float(auc_drift_day_router),
            base_confusion_matrix=cm_base.tolist(),
            joint_confusion_matrix=cm_joint.tolist(),
            router_coef=router_clf.coef_.tolist(),
            router_intercept=router_clf.intercept_.tolist(),
        )
        metrics.update(prefix_keys(base_metrics, "base_"))
        metrics.update(prefix_keys(joint_metrics, "joint_"))

        # deltas
        for key in [
            "stable_acc_gate",
            "drift_acc_all",
            "unknown_reject_all",
            "FRR_stable",
            "FAR_unknown_accept",
            "false_unknown_drift_all",
            "false_drift_unknown",
        ]:
            metrics[f"delta_{key}"] = float(metrics[f"joint_{key}"] - metrics[f"base_{key}"])

        with open(os.path.join(fold_dir, "metrics_joint_router.json"), "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)

        rows.append(metrics)
        print(
            f"[{protocol_tag} split {split_id} fold {fold}] "
            f"base(drift={metrics['base_drift_acc_all']:.3f}, unkrej={metrics['base_unknown_reject_all']:.3f}) | "
            f"joint(drift={metrics['joint_drift_acc_all']:.3f}, unkrej={metrics['joint_unknown_reject_all']:.3f}) | "
            f"deltas(drift={metrics['delta_drift_acc_all']:+.3f}, unkrej={metrics['delta_unknown_reject_all']:+.3f})"
        )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for ps in PROTOCOL_SPECS:
    protocol_tag = ps["protocol_tag"]
    print("\n==============================")
    print("Protocol:", protocol_tag)
    print("SOURCE_RXS:", ps["source_rxs"])
    print("DRIFT_RX:", ps["drift_rx"])
    print("==============================")
    for split_pos, tx_split in enumerate(ps["tx_splits"], start=1):
        if isinstance(tx_split, dict):
            split_id = int(tx_split.get("split_id", split_pos))
            KNOWN_TX = list(tx_split["known"])
            UNKNOWN_TX = list(tx_split["unknown"])
        else:
            split_id = split_pos
            KNOWN_TX, UNKNOWN_TX = tx_split
            KNOWN_TX = list(KNOWN_TX)
            UNKNOWN_TX = list(UNKNOWN_TX)

        print("\n=== TX split", split_id, "===")
        print("KNOWN_TX:", KNOWN_TX)
        print("UNKNOWN_TX:", UNKNOWN_TX)
        all_rows.extend(run_one_split(
            protocol_tag=protocol_tag,
            rx_protocol_tag=ps["rx_protocol"],
            tx_protocol_tag=ps["tx_protocol"],
            split_id=split_id,
            KNOWN_TX=KNOWN_TX,
            UNKNOWN_TX=UNKNOWN_TX,
            source_rxs=ps["source_rxs"],
            drift_rx=ps["drift_rx"],
        ))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)


Protocol: RX9-3_TX4-2
SOURCE_RXS: [np.str_('14-7'), np.str_('18-2'), np.str_('2-1'), np.str_('2-19'), np.str_('20-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7'), np.str_('8-8')]
DRIFT_RX: ['1-1', '1-19', '19-2']

=== TX split 1 ===
KNOWN_TX: ['14-7', '20-15', '20-19', '8-20']
UNKNOWN_TX: ['14-10', '6-15']


c:\Users\HP\anaconda3\envs\MW-RFF\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[RX9-3_TX4-2 split 1 fold 1] base(drift=0.079, unkrej=0.743) | joint(drift=0.431, unkrej=0.646) | deltas(drift=+0.352, unkrej=-0.097)


c:\Users\HP\anaconda3\envs\MW-RFF\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[RX9-3_TX4-2 split 1 fold 2] base(drift=0.126, unkrej=0.501) | joint(drift=0.311, unkrej=0.567) | deltas(drift=+0.184, unkrej=+0.067)


...

[RX3-9_TX2-4 split 5 fold 3] base(drift=0.258, unkrej=0.958) | joint(drift=0.584, unkrej=0.668) | deltas(drift=+0.326, unkrej=-0.290)


c:\Users\HP\anaconda3\envs\MW-RFF\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[RX3-9_TX2-4 split 5 fold 4] base(drift=0.129, unkrej=0.721) | joint(drift=0.484, unkrej=0.576) | deltas(drift=+0.355, unkrej=-0.145)


c:\Users\HP\anaconda3\envs\MW-RFF\lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[RX3-9_TX2-4 split 5 fold 5] base(drift=0.086, unkrej=0.838) | joint(drift=0.595, unkrej=0.676) | deltas(drift=+0.510, unkrej=-0.163)

Saved all metrics to: ../owdc_results/results_wisig_module2_joint_router_protocols/run_20260324_142641


In [10]:
# =========================
# Summary
# =========================
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

SUMMARY_KEYS = [
    "acc",
    "auc_unknown_sid",
    "auc_unknown_jointprob",
    "auc_drift_rx_raw_hard",
    "auc_drift_day_raw_hard",
    "auc_drift_rx_router",
    "auc_drift_day_router",
    "base_stable_acc_gate",
    "base_drift_acc_all",
    "base_unknown_reject_all",
    "base_FRR_stable",
    "base_FAR_unknown_accept",
    "base_false_unknown_drift_all",
    "joint_stable_acc_gate",
    "joint_drift_acc_all",
    "joint_unknown_reject_all",
    "joint_FRR_stable",
    "joint_FAR_unknown_accept",
    "joint_false_unknown_drift_all",
    "delta_stable_acc_gate",
    "delta_drift_acc_all",
    "delta_unknown_reject_all",
    "delta_FRR_stable",
    "delta_FAR_unknown_accept",
    "delta_false_unknown_drift_all",
    "tau_id",
    "tau_dom",
]

summary_overall = {}
for key in SUMMARY_KEYS:
    m, s = mean_std([r[key] for r in all_rows])
    summary_overall[key] = {"mean": m, "std": s}

cm_base_sum = np.zeros((3,3), dtype=np.int64)
cm_joint_sum = np.zeros((3,3), dtype=np.int64)
for r in all_rows:
    cm_base_sum += np.array(r["base_confusion_matrix"], dtype=np.int64)
    cm_joint_sum += np.array(r["joint_confusion_matrix"], dtype=np.int64)
summary_overall["base_confusion_matrix_sum"] = cm_base_sum.tolist()
summary_overall["joint_confusion_matrix_sum"] = cm_joint_sum.tolist()

summary_by_protocol = {}
protocol_tags = sorted(set(r["protocol_tag"] for r in all_rows))
for ptag in protocol_tags:
    rows_p = [r for r in all_rows if r["protocol_tag"] == ptag]
    block = {"n_rows": len(rows_p)}
    for key in SUMMARY_KEYS:
        m, s = mean_std([r[key] for r in rows_p])
        block[key] = {"mean": m, "std": s}
    cm_b = np.zeros((3,3), dtype=np.int64)
    cm_j = np.zeros((3,3), dtype=np.int64)
    for r in rows_p:
        cm_b += np.array(r["base_confusion_matrix"], dtype=np.int64)
        cm_j += np.array(r["joint_confusion_matrix"], dtype=np.int64)
    block["base_confusion_matrix_sum"] = cm_b.tolist()
    block["joint_confusion_matrix_sum"] = cm_j.tolist()
    summary_by_protocol[ptag] = block

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary_overall, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_protocol, f, indent=2)

print("=== Overall Mean ± Std over protocols × splits × folds ===")
for k, v in summary_overall.items():
    if "confusion_matrix_sum" in k:
        continue
    print(f"{k:26s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\nBase confusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:")
print(cm_base_sum)
print("\nJoint confusion matrix SUM (rows=GT, cols=Pred) [Stable,Drift,Unknown]:")
print(cm_joint_sum)

print("\n=== Per-protocol summary ===")
for ptag, block in summary_by_protocol.items():
    print(f"\n[{ptag}] n_rows={block['n_rows']}")
    for key in [
        "base_drift_acc_all",
        "base_unknown_reject_all",
        "joint_drift_acc_all",
        "joint_unknown_reject_all",
        "delta_drift_acc_all",
        "delta_unknown_reject_all",
        "delta_FRR_stable",
        "delta_FAR_unknown_accept",
    ]:
        v = block[key]
        print(f"  {key:26s}: {v['mean']:.3f} ± {v['std']:.3f}")

=== Overall Mean ± Std over protocols × splits × folds ===
acc                       : 0.986 ± 0.052
auc_unknown_sid           : 0.873 ± 0.093
auc_unknown_jointprob     : 0.957 ± 0.028
auc_drift_rx_raw_hard     : 0.872 ± 0.049
auc_drift_day_raw_hard    : 0.711 ± 0.046
auc_drift_rx_router       : 0.906 ± 0.050
auc_drift_day_router      : 0.726 ± 0.050
base_stable_acc_gate      : 0.902 ± 0.015
base_drift_acc_all        : 0.173 ± 0.093
base_unknown_reject_all   : 0.666 ± 0.174
base_FRR_stable           : 0.098 ± 0.015
base_FAR_unknown_accept   : 0.334 ± 0.174
base_false_unknown_drift_all: 0.347 ± 0.182
joint_stable_acc_gate     : 0.822 ± 0.067
joint_drift_acc_all       : 0.469 ± 0.108
joint_unknown_reject_all  : 0.742 ± 0.087
joint_FRR_stable          : 0.178 ± 0.067
joint_FAR_unknown_accept  : 0.258 ± 0.087
joint_false_unknown_drift_all: 0.254 ± 0.106
delta_stable_acc_gate     : -0.080 ± 0.071
delta_drift_acc_all       : 0.295 ± 0.100
delta_unknown_reject_all  : 0.076 ± 0.175
delta_FRR_s